## Orchestrator

Coordinates the full ingestion pipeline by importing functions from the other notebooks and chaining them:

```
extract (bronze)  ->  load raw payload  ->  transform (silver)  ->  load typed table
```

Two entry points are defined:
- `run_ohlcv(symbols)` - daily run for OHLCV data
- `run_overview(symbols)` - weekly run for company fundamentals

OHLCV and Overview are kept on separate schedules to preserve the 25 requests/day quota of the Alpha Vantage free tier.

### Setup

Imports functions from the three sibling notebooks using `import_ipynb`.

Since notebook filenames start with a digit (not a valid Python identifier), `importlib.import_module` is used instead of a direct `import` statement.

Install dependency if needed: `pip install import-ipynb`

In [ ]:
import time
import importlib

import import_ipynb  # noqa: F401 - registers the .ipynb import hook

ohlcv_nb = importlib.import_module("01_ohlcv")
overview_nb = importlib.import_module("02_overview")
db = importlib.import_module("03_database_connection")

### Configuration

`sleep_seconds` controls the delay between API calls. The Alpha Vantage free tier has a daily limit (25 req/day), not a per-minute one, so a small sleep is just good citizenship - not strictly required.

In [ ]:
symbols = ["IBM", "AAPL", "MSFT", "GOOGL", "AMZN"]
sleep_seconds = 15

### OHLCV Pipeline (daily)

For each symbol:
1. Fetch raw payload from Alpha Vantage (`ohlcv_bronze`)
2. Store raw payload in `market.raw_ohlcv` (`load_raw_ohlcv`)
3. Transform payload into DataFrame (`ohlcv_silver`)
4. Upsert DataFrame into `market.ohlcv` (`load_ohlcv`)

Errors on a single symbol are caught and logged - the pipeline continues with the remaining tickers. The function returns a summary dict so failed loads can be inspected after the run.

In [ ]:
def run_ohlcv(symbols: list[str]) -> dict:
    results = {"ok": [], "failed": []}

    for i, symbol in enumerate(symbols):
        try:
            payload = ohlcv_nb.ohlcv_bronze(symbol, ohlcv_nb.api_key)
            db.load_raw_ohlcv(symbol, payload)

            df = ohlcv_nb.ohlcv_silver(payload)
            rows = db.load_ohlcv(df)

            print(f"[{i + 1}/{len(symbols)}] {symbol} OK - {rows} rows")
            results["ok"].append({"symbol": symbol, "rows": rows})

        except Exception as e:
            print(f"[{i + 1}/{len(symbols)}] {symbol} FAILED: {e}")
            results["failed"].append({"symbol": symbol, "error": str(e)})

        if i < len(symbols) - 1:
            time.sleep(sleep_seconds)

    return results

### Overview Pipeline (weekly)

Same pattern as OHLCV, but using the Overview endpoint functions. Intended to run on a weekly schedule to preserve API quota - fundamentals don't change daily.

In [ ]:
def run_overview(symbols: list[str]) -> dict:
    results = {"ok": [], "failed": []}

    for i, symbol in enumerate(symbols):
        try:
            payload = overview_nb.overview_bronze(symbol, overview_nb.api_key)
            db.load_raw_overview(symbol, payload)

            df = overview_nb.overview_silver(payload)
            rows = db.load_overview(df)

            print(f"[{i + 1}/{len(symbols)}] {symbol} OK - {rows} rows")
            results["ok"].append({"symbol": symbol, "rows": rows})

        except Exception as e:
            print(f"[{i + 1}/{len(symbols)}] {symbol} FAILED: {e}")
            results["failed"].append({"symbol": symbol, "error": str(e)})

        if i < len(symbols) - 1:
            time.sleep(sleep_seconds)

    return results

### Execution

Run the cells below to trigger the pipelines. Each run is independent - pick the one that matches your schedule (OHLCV daily, Overview weekly).

In [ ]:
ohlcv_results = run_ohlcv(symbols)
ohlcv_results

In [ ]:
overview_results = run_overview(symbols)
overview_results